In [3]:
import sys
sys.path.append('../')

import numpy as np
from openfermion import QubitOperator, get_sparse_operator
from numpy.random import uniform
from utils_m2_factorize import (
    expand_tensor_product, 
    expand_tensor_product_for_incomplete_qubit_set,
    partition_from_dict,
    obtain_join_partition,
    obtain_coarse_dicts,
    QC_assignment_from_qubit_labels,
    split_pauli_operator,
    relabel_qubits_in_pauli_term,
    evaluate_matrix_element_given_term,
    partially_evaluate_pauli_term,
    partially_evaluate_hamiltonian_matrix_element,
    evaluate_fully_classical_factors
)
from utils_basic import (
    random_pauli_hamiltonian
)
from utils_states import (
    convert_dense_format_to_sparse_format
)

verify `split_pauli_operator`

In [4]:
P = QubitOperator('X0 X1 Y2 Y3 Z6 Z7 X8 Y9')

block  = (0,1,2,3)
P1, P2 = split_pauli_operator(P, block)
print(block, '\n', QubitOperator(P1), '\n', QubitOperator(P2), '\n')
assert QubitOperator(P1) * QubitOperator(P2) == P

block  = (4,5)
P1, P2 = split_pauli_operator(P, block)
print(block, '\n', QubitOperator(P1), '\n', QubitOperator(P2), '\n')
assert QubitOperator(P1) * QubitOperator(P2) == P

block  = (0,3,4,8)
P1, P2 = split_pauli_operator(P, block)
print(block, '\n', QubitOperator(P1), '\n', QubitOperator(P2), '\n')
assert QubitOperator(P1) * QubitOperator(P2) == P


(0, 1, 2, 3) 
 1.0 [X0 X1 Y2 Y3] 
 1.0 [Z6 Z7 X8 Y9] 

(4, 5) 
 1.0 [] 
 1.0 [X0 X1 Y2 Y3 Z6 Z7 X8 Y9] 

(0, 3, 4, 8) 
 1.0 [X0 Y3 X8] 
 1.0 [X1 Y2 Z6 Z7 Y9] 



verify `relabel_qubits_in_pauli_term`

In [5]:
block  = (0,1,2,3)
P1, P2 = split_pauli_operator(P, block)
print(QubitOperator(P1))
print(QubitOperator(relabel_qubits_in_pauli_term(P1, block)))
print()

block  = (4,5)
P1, P2 = split_pauli_operator(P, block)
print(QubitOperator(P1))
print(QubitOperator(relabel_qubits_in_pauli_term(P1, block)))
print()

block  = (0,3,4,8)
P1, P2 = split_pauli_operator(P, block)
print(QubitOperator(P1))
print(QubitOperator(relabel_qubits_in_pauli_term(P1, block)))

1.0 [X0 X1 Y2 Y3]
1.0 [X0 X1 Y2 Y3]

1.0 []
1.0 []

1.0 [X0 Y3 X8]
1.0 [X0 Y1 X3]


Test 1 of `partially_evaluate_hamiltonian_matrix_element`:

Given a factorizable state

$$
|\psi\rangle = |\psi_{012}\rangle \otimes |\psi_{345}\rangle,
$$

two evaluations of `partially_evaluate_hamiltonian_matrix_element` for some random initial Hamiltonian should recover the correct expectation value.

In [6]:
Nqubits = 6
Nterms  = 200

H       = random_pauli_hamiltonian(Nqubits, Nterms)
Hsp     = get_sparse_operator(H, Nqubits)

psi012  = uniform(-1, 1, 2**3)
psi345  = uniform(-1, 1, 2**3)
psi     = np.kron(psi012, psi345)

expectation       = (psi @ Hsp @ psi)

H345              = partially_evaluate_hamiltonian_matrix_element(psi012, psi012, H, (0,1,2))
expectation_recov = partially_evaluate_hamiltonian_matrix_element(psi345, psi345, H345, (3,4,5))

assert expectation_recov - expectation == QubitOperator().zero()

H012 = partially_evaluate_hamiltonian_matrix_element(psi345, psi345, H, (3,4,5))
expectation_recov = partially_evaluate_hamiltonian_matrix_element(psi012, psi012, H012, (0,1,2))

assert expectation_recov - expectation == QubitOperator().zero()

Test 2 of `partially_evaluate_hamiltonian_matrix_element`: Same test as test 1 but now it is an off-diagonal matrix element. There are two examples in what follows.



In [7]:
for _ in range(10):

    Nqubits = 10
    Nterms = 500

    H   = random_pauli_hamiltonian(Nqubits, Nterms)
    Hsp = get_sparse_operator(H, Nqubits)

    psi0123456 = uniform(-1, 1, 2**7)
    psi789     = uniform(-1, 1, 2**3)
    psi        = np.kron(psi0123456, psi789)

    phi0123456 = uniform(-1, 1, 2**7)
    phi789     = uniform(-1, 1, 2**3)
    phi        = np.kron(phi0123456, phi789)

    expectation = phi @ Hsp @ psi

    H789 = partially_evaluate_hamiltonian_matrix_element(phi0123456, psi0123456, H, (0,1,2,3,4,5,6))
    expectation_recov1 = partially_evaluate_hamiltonian_matrix_element(phi789, psi789, H789, (7,8,9))

    H0123456 = partially_evaluate_hamiltonian_matrix_element(phi789, psi789, H, (7,8,9))
    expectation_recov2 = partially_evaluate_hamiltonian_matrix_element(phi0123456, psi0123456, H0123456, (0,1,2,3,4,5,6))

    assert expectation - expectation_recov1 == QubitOperator().zero()
    assert expectation - expectation_recov2 == QubitOperator().zero()

In [8]:
for _ in range(10):

    Nqubits = 12
    Nterms = 1000

    H = random_pauli_hamiltonian(Nqubits, Nterms)
    Hsp = get_sparse_operator(H, Nqubits)

    psi01234    = uniform(-1, 1, 2**5)
    psi567891011 = uniform(-1, 1, 2**7)
    psi         = np.kron(psi01234, psi567891011)

    phi01234    = uniform(-1, 1, 2**5)
    phi567891011 = uniform(-1, 1, 2**7)
    phi         = np.kron(phi01234, phi567891011)

    element = psi @ Hsp @ phi

    H01234    = partially_evaluate_hamiltonian_matrix_element(psi567891011, phi567891011, H, (5,6,7,8,9,10,11))
    element1  = partially_evaluate_hamiltonian_matrix_element(psi01234, phi01234, H01234, (0,1,2,3,4))

    H67891011 = partially_evaluate_hamiltonian_matrix_element(psi01234, phi01234, H, (0,1,2,3,4))
    element2  = partially_evaluate_hamiltonian_matrix_element(psi567891011, phi567891011, H67891011, (5,6,7,8,9,10,11))

    assert element - element1 == QubitOperator().zero()
    assert element - element2 == QubitOperator().zero()

Test 3 of `partially_evaluate_hamiltonian_matrix_element`: Here the tensor products are non-adjacent.

In [9]:
for _ in range(10):

    Nqubits = 10
    Nterms  = 500

    H   = random_pauli_hamiltonian(Nqubits, Nterms)
    Hsp = get_sparse_operator(H, Nqubits)

    psi0345   = uniform(-1, 1, 2**4)
    psi126789 = uniform(-1, 1, 2**6)
    psi_dict  = {
        (0,3,4,5)     : psi0345,
        (1,2,6,7,8,9) : psi126789
    }
    psi       = expand_tensor_product_for_incomplete_qubit_set(psi_dict)

    phi0345   = uniform(-1, 1, 2**4)
    phi126789 = uniform(-1, 1, 2**6)
    phi_dict  = {
        (0,3,4,5)     : phi0345,
        (1,2,6,7,8,9) : phi126789
    }
    phi       = expand_tensor_product_for_incomplete_qubit_set(phi_dict)


    element = psi @ Hsp @ phi

    H126789  = partially_evaluate_hamiltonian_matrix_element(psi0345, phi0345, H, (0,3,4,5))
    element2 = partially_evaluate_hamiltonian_matrix_element(psi126789, phi126789, H126789, (1,2,6,7,8,9))

    H0345    = partially_evaluate_hamiltonian_matrix_element(psi126789, phi126789, H, (1,2,6,7,8,9))
    element3 = partially_evaluate_hamiltonian_matrix_element(psi0345, phi0345, H0345, (0,3,4,5))

    assert element - element2 == QubitOperator().zero()
    assert element - element2 == QubitOperator().zero()

In [10]:
for _ in range(10):

    Nqubits = 10
    Nterms = 500

    H   = random_pauli_hamiltonian(Nqubits, Nterms)
    Hsp = get_sparse_operator(H, Nqubits)

    psi04    = uniform(-1, 1, 2**2)
    psi5789  = uniform(-1, 1, 2**4)
    psi1236  = uniform(-1, 1, 2**4)
    psi_dict = {
        (0,4)     : psi04,
        (5,7,8,9) : psi5789,
        (1,2,3,6) : psi1236
    }
    psi      = expand_tensor_product_for_incomplete_qubit_set(psi_dict)

    phi04    = uniform(-1, 1, 2**2)
    phi5789  = uniform(-1, 1, 2**4)
    phi1236  = uniform(-1, 1, 2**4)
    phi_dict = {
        (0,4)     : phi04,
        (5,7,8,9) : phi5789,
        (1,2,3,6) : phi1236
    }
    phi      = expand_tensor_product_for_incomplete_qubit_set(phi_dict)

    element = psi @ Hsp @ phi

    H1 = partially_evaluate_hamiltonian_matrix_element(psi04, phi04, H, (0,4))
    H2 = partially_evaluate_hamiltonian_matrix_element(psi5789, phi5789, H1, (5,7,8,9))
    element2 = partially_evaluate_hamiltonian_matrix_element(psi1236, phi1236, H2, (1,2,3,6))

    assert element - element2 == QubitOperator().zero()

    H1 = partially_evaluate_hamiltonian_matrix_element(psi1236, phi1236, H, (1,2,3,6))
    H2 = partially_evaluate_hamiltonian_matrix_element(psi5789, phi5789, H1, (5,7,8,9))
    element3 = partially_evaluate_hamiltonian_matrix_element(psi04, phi04, H2, (0,4))

    assert element - element3 == QubitOperator().zero()

    H1 = partially_evaluate_hamiltonian_matrix_element(psi5789, phi5789, H, (5,7,8,9))
    H2 = partially_evaluate_hamiltonian_matrix_element(psi04, phi04, H1, (0,4))
    element4 = partially_evaluate_hamiltonian_matrix_element(psi1236, phi1236, H2, (1,2,3,6))

    assert element - element4 == QubitOperator().zero()

Test `evaluate_fully_classical_factors`.

The following tests are done to verify if this function works.

1. The matrix element is unchanged by this function.
2. The output quantum states correspond to the tensor product of `Q` parts for the original quantum states.

In [ ]:
p    = [(0,4), (5,7,8,9), (1,2,3,6)]
join = obtain_join_partition(p, p)

psi_labels = {
    0 : 'V',
    1 : 'V',
    2 : 'V',
    3 : 'W',
    4 : 'W',
    5 : 'N',
    6 : 'N',
    7 : 'N',
    8 : 'V',
    9 : 'V',
}

phi_labels = {
    0 : 'V',
    1 : 'V',
    2 : 'W',
    3 : 'N',
    4 : 'V',
    5 : 'N',
    6 : 'W',
    7 : 'N',
    8 : 'V',
    9 : 'V',
}

print("Qubit Assignment")
print(QC_assignment_from_qubit_labels(phi_labels, psi_labels, join))

for _ in range(10):
    Nqubits = 10
    Nterms = 500

    H   = random_pauli_hamiltonian(Nqubits, Nterms)
    H  += uniform(-3, 3) * QubitOperator("")
    Hsp = get_sparse_operator(H, Nqubits)

    psi04      = uniform(-1, 1, 2**2)
    psi5789    = uniform(-1, 1, 2**4)
    psi1236    = uniform(-1, 1, 2**4)
    psi_dict   = {
        (0,4)     : psi04,
        (5,7,8,9) : psi5789,
        (1,2,3,6) : psi1236
    }
    psi        = expand_tensor_product_for_incomplete_qubit_set(psi_dict)
    phi04      = uniform(-1, 1, 2**2)
    phi5789    = uniform(-1, 1, 2**4)
    phi1236    = uniform(-1, 1, 2**4)
    phi_dict   = {
        (0,4)     : phi04,
        (5,7,8,9) : phi5789,
        (1,2,3,6) : phi1236
    }
    phi        = expand_tensor_product_for_incomplete_qubit_set(phi_dict)

    Heff, psiQ, phiQ, NqubitsQ = evaluate_fully_classical_factors(psi_dict, phi_dict, psi_labels, phi_labels, H)
    Heffsp                     = get_sparse_operator(Heff, NqubitsQ)
    assert NqubitsQ == 6

    # matrix element is preserved
    element1 = psi @ Hsp @ phi
    element2 = psiQ @ Heffsp @ phiQ

    assert np.abs(element1 - element2) < 1e-12

    # output states correspond to quantum parts of input states
    psi_quantum_dict            = {}
    psi_quantum_dict[(0,4)]     = psi_dict[(0,4)]
    psi_quantum_dict[(1,2,3,6)] = psi_dict[(1,2,3,6)]
    
    phi_quantum_dict            = {}
    phi_quantum_dict[(0,4)]     = phi_dict[(0,4)]
    phi_quantum_dict[(1,2,3,6)] = phi_dict[(1,2,3,6)]

    assert np.allclose(
        psiQ, expand_tensor_product_for_incomplete_qubit_set(psi_quantum_dict)
    )
    assert np.allclose(
        phiQ, expand_tensor_product_for_incomplete_qubit_set(phi_quantum_dict)
    )
    assert np.shape(phiQ)[0] == 2 ** 6
    assert np.shape(psiQ)[0] == 2 ** 6

Qubit Assignment
{(0, 4): 'Q', (1, 2, 3, 6): 'Q', (5, 7, 8, 9): 'C'}


In [ ]:
# same test as above but now the states factorize over different subsets of qubits

Nqubits = 13
Nterms = 500

bra_partition = [(0,1,2), (3,4,5), (6,), (7,), (8,), (9,10), (11,12)]
ket_partition = [(0,1), (2,3,4,5), (6,7), (8,), (9,10,11), (12,)]

bra_labels = {
    0  : 'W',
    1  : 'W',
    2  : 'W',
    3  : 'V',
    4  : 'V',
    5  : 'V',
    6  : 'V',
    7  : 'N',
    8  : 'N',
    9  : 'V',
    10 : 'V',
    11 : 'N',
    12 : 'N'
}

ket_labels = {
    0  : 'W',
    1  : 'W',
    2  : 'V',
    3  : 'V',
    4  : 'V',
    5  : 'V',
    6  : 'N',
    7  : 'N',
    8  : 'W',
    9  : 'V',
    10 : 'V',
    11 : 'V',
    12 : 'N'
}

join_partition = obtain_join_partition(bra_partition, ket_partition)
QC_assignment  = QC_assignment_from_qubit_labels(bra_labels, ket_labels, join_partition)

print("Qubit Assignment")
print(QC_assignment)

for _ in range(10):
    H   = random_pauli_hamiltonian(Nqubits, Nterms)
    H  += uniform(-3, 3) * QubitOperator("")
    Hsp = get_sparse_operator(H, Nqubits)

    bra012   = uniform(-1, 1, 2**3)
    bra345   = uniform(-1, 1, 2**3)
    bra6     = uniform(-1, 1, 2**1)
    bra7     = uniform(-1, 1, 2**1)
    bra8     = uniform(-1, 1, 2**1)
    bra910   = uniform(-1, 1, 2**2)
    bra1112  = uniform(-1, 1, 2**2)
    bra_dict = {
        bra_partition[0] : bra012,
        bra_partition[1] : bra345,
        bra_partition[2] : bra6,
        bra_partition[3] : bra7,
        bra_partition[4] : bra8,
        bra_partition[5] : bra910,
        bra_partition[6] : bra1112
    }
    bra      = expand_tensor_product_for_incomplete_qubit_set(bra_dict)

    ket01    = uniform(-1, 1, 2**2)
    ket2345  = uniform(-1, 1, 2**4)
    ket67    = uniform(-1, 1, 2**2)
    ket8     = uniform(-1, 1, 2**1)
    ket91011 = uniform(-1, 1, 2**3)
    ket12    = uniform(-1, 1, 2**1)
    ket_dict = {
        ket_partition[0] : ket01,
        ket_partition[1] : ket2345,
        ket_partition[2] : ket67,
        ket_partition[3] : ket8,
        ket_partition[4] : ket91011,
        ket_partition[5] : ket12
    }
    ket      = expand_tensor_product_for_incomplete_qubit_set(ket_dict)

    Heff, braQ, ketQ, NqubitsQ = evaluate_fully_classical_factors(bra_dict, ket_dict, bra_labels, ket_labels, H)
    Heffsp                     = get_sparse_operator(Heff, 7)
    assert NqubitsQ == 7
    
    # check that the matrix element is preserved by the transformation
    element1 = bra @ Hsp @ ket
    element2 = (convert_dense_format_to_sparse_format(braQ) @ Heffsp @ convert_dense_format_to_sparse_format(ketQ).T)[0,0]

    assert np.abs(element1 - element2) < 1e-12

    # check that the states are constructed correctly
    braQ_dict          = {}
    braQ_dict[(0,1,2)] = bra_dict[(0,1,2)]
    braQ_dict[(3,4,5)] = bra_dict[(3,4,5)]
    braQ_dict[(8,)]    = bra_dict[(8,)]

    ketQ_dict            = {}
    ketQ_dict[(0,1)]     = ket_dict[(0,1)]
    ketQ_dict[(2,3,4,5)] = ket_dict[(2,3,4,5)]
    ketQ_dict[(8,)]      = ket_dict[(8,)]

    assert np.allclose(
        braQ, expand_tensor_product_for_incomplete_qubit_set(braQ_dict)
    )
    assert np.allclose(
        ketQ, expand_tensor_product_for_incomplete_qubit_set(ketQ_dict)
    )

    assert np.shape(braQ)[0] == 2 ** 7
    assert np.shape(ketQ)[0] == 2 ** 7

Qubit Assignment
{(0, 1, 2, 3, 4, 5): 'Q', (6, 7): 'C', (8,): 'Q', (9, 10, 11, 12): 'C'}


In [ ]:
# same test as above but now all states are classically evaluated

Nqubits = 13
Nterms = 500

bra_partition = [(0,1,2), (3,4,5), (6,), (7,), (8,), (9,10), (11,12)]
ket_partition = [(0,1), (2,3,4,5), (6,7), (8,), (9,10,11), (12,)]

bra_labels = {
    0  : 'N',
    1  : 'N',
    2  : 'N',
    3  : 'N',
    4  : 'N',
    5  : 'N',
    6  : 'N',
    7  : 'N',
    8  : 'N',
    9  : 'N',
    10 : 'N',
    11 : 'N',
    12 : 'N'
}

ket_labels = {
    0  : 'N',
    1  : 'N',
    2  : 'N',
    3  : 'N',
    4  : 'N',
    5  : 'N',
    6  : 'N',
    7  : 'N',
    8  : 'N',
    9  : 'N',
    10 : 'N',
    11 : 'N',
    12 : 'N'
}

join_partition = obtain_join_partition(bra_partition, ket_partition)
QC_assignment  = QC_assignment_from_qubit_labels(bra_labels, ket_labels, join_partition)

print("Qubit Assignment")
print(QC_assignment)

for _ in range(10):
    H   = random_pauli_hamiltonian(Nqubits, Nterms)
    H  += uniform(-3, 3) * QubitOperator("")
    Hsp = get_sparse_operator(H, Nqubits)

    bra012   = uniform(-1, 1, 2**3)
    bra345   = uniform(-1, 1, 2**3)
    bra6     = uniform(-1, 1, 2**1)
    bra7     = uniform(-1, 1, 2**1)
    bra8     = uniform(-1, 1, 2**1)
    bra910   = uniform(-1, 1, 2**2)
    bra1112  = uniform(-1, 1, 2**2)
    bra_dict = {
        bra_partition[0] : bra012,
        bra_partition[1] : bra345,
        bra_partition[2] : bra6,
        bra_partition[3] : bra7,
        bra_partition[4] : bra8,
        bra_partition[5] : bra910,
        bra_partition[6] : bra1112
    }
    bra      = expand_tensor_product_for_incomplete_qubit_set(bra_dict)

    ket01    = uniform(-1, 1, 2**2)
    ket2345  = uniform(-1, 1, 2**4)
    ket67    = uniform(-1, 1, 2**2)
    ket8     = uniform(-1, 1, 2**1)
    ket91011 = uniform(-1, 1, 2**3)
    ket12    = uniform(-1, 1, 2**1)
    ket_dict = {
        ket_partition[0] : ket01,
        ket_partition[1] : ket2345,
        ket_partition[2] : ket67,
        ket_partition[3] : ket8,
        ket_partition[4] : ket91011,
        ket_partition[5] : ket12
    }
    ket      = expand_tensor_product_for_incomplete_qubit_set(ket_dict)

    Heff, braQ, ketQ, NqubitsQ = evaluate_fully_classical_factors(bra_dict, ket_dict, bra_labels, ket_labels, H)
    Heffsp                     = get_sparse_operator(Heff, 7)
    assert NqubitsQ == 0
    
    # check that the matrix element is preserved by the transformation
    element1 = bra @ Hsp @ ket
    element2 = Heff.constant

    assert np.abs(element1 - element2) < 1e-8


Qubit Assignment
{(0, 1, 2, 3, 4, 5): 'C', (6, 7): 'C', (8,): 'C', (9, 10, 11, 12): 'C'}
(-0.02160149876377039+0.3196544668591642j) (-0.021601498763770682+0.3196544668591641j)
(0.1385941655608579-0.14782930973628683j) (0.13859416556085769-0.147829309736286j)
(0.16637513817035296+0.16076795487254983j) (0.16637513817035285+0.16076795487254983j)
(-0.44252005418239465+0.028316032480750153j) (-0.44252005418239404+0.028316032480749043j)
(0.02454923271686626-0.015661880359150927j) (0.024549232716866244-0.01566188035915078j)
(2.2103555564914084-3.0225739976342094j) (2.210355556491399-3.022573997634213j)
(0.6185874165065178-0.22407664968829621j) (0.6185874165065173-0.22407664968829513j)
(-0.8155150605292905+0.9354542821223626j) (-0.8155150605292953+0.935454282122364j)
(0.8286172928879687+0.29364670257904957j) (0.8286172928879709+0.29364670257904935j)
(-0.811662228461511+0.4387948539264137j) (-0.81166222846151+0.43879485392641343j)
